# HTML Schedule Export
This notebook parametrises the HTML/Jinja pipeline to generate branded schedules for classes, teachers (_pros_), and rooms (_locals_).

- Use the `RENDERER` parameter below to pick `wkhtmltopdf` (good default) or `chromium` (requires a local Chrome/Chromium binary) when you want Chrome-quality PDFs.
- The notebook also writes responsive HTML pages alongside PDFs so you can open them in a browser and print at 100% if needed.


In [16]:
from pathlib import Path

# --- Pipeline parameters ---
INPUT_FILE = Path("../input/inputSM_2025-2026_3_10.xlsx")
SHEET_NAME = "input"
SEMVER = "2025-2026-v0.9"
TEMPLATE_ROOT = Path("../templates/html")
TEMPLATE_NAME = "schedule.html.j2"
OUTPUT_DIR = Path("../output/html")
INCLUDE_GENERAL = True  # Include classe entière rows when rendering subgroup sheets
RENDERER = "chromium"  # pdf renderer: weasyprint, wkhtmltopdf, or chromium


In [17]:
import shutil
from shutil import which
print("wkhtmltopdf:", which("wkhtmltopdf"))


wkhtmltopdf: /usr/bin/wkhtmltopdf


In [18]:
import pandas as pd

df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
display(df.head())

,classe,#day,day,plage,start,end,semaine,cours,pro,local classe,local,code horaire,code,groupe,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,1A,1,Lundi,P1,8h45,10h15,AB,Français,pauline.crochet,"-1,06","-1,06",1A,FRA1C,NaN,NaN,NaN,NaN
1,1A,1,Lundi,P2,10h30,12h,AB,Éducation aux médias,jean.matterne,"-1,06","-1,06",1A,EMI1C,NaN,NaN,NaN,NaN
2,1A,1,Lundi,P3,13h15,14h45,AB,Mathématiques,souhaib.darraz,"-1,06","-1,06",1A,MAT1C,NaN,NaN,NaN,NaN
3,1A,1,Lundi,P4,15h,16h30,A,Sciences,juliette.alarcia,"-1,06","-1,06",1A,SCI1C,NaN,NaN,NaN,NaN
4,1A,1,Lundi,P4,15h,16h30,B,Éducation physique,christophe.guenard,"-1,06","-1,06",1A,EP1C,NaN,NaN,NaN,NaN


In [19]:
from fragenda.html_export import generate_class_pdfs, generate_scope_pdfs

OUTPUT_DIR_CLASSES = OUTPUT_DIR / "classes"
OUTPUT_DIR_PROS = OUTPUT_DIR / "pros"
OUTPUT_DIR_LOCALS = OUTPUT_DIR / "locals"


## Classes

In [20]:
class_pdfs = generate_class_pdfs(
    dataframe=df,
    semver=SEMVER,
    template_dir=TEMPLATE_ROOT,
    template_name=TEMPLATE_NAME,
    output_dir=OUTPUT_DIR_CLASSES,
    include_general=INCLUDE_GENERAL,
    renderer=RENDERER,
)
print(f"Generated {len(class_pdfs)} class schedule PDFs → {OUTPUT_DIR_CLASSES.resolve()}")

Classes:   0%|          | 0/55 [00:00<?, ?doc/s]

Generated 55 class schedule PDFs → /home/oscar/Projects/smartschool/fragenda/output/html/classes


## Pros (Teachers)

In [21]:
pro_pdfs = generate_scope_pdfs(
    dataframe=df,
    scope="pro",
    semver=SEMVER,
    template_dir=TEMPLATE_ROOT,
    template_name=TEMPLATE_NAME,
    output_dir=OUTPUT_DIR_PROS,
    renderer=RENDERER,
)
print(f"Generated {len(pro_pdfs)} teacher schedule PDFs → {OUTPUT_DIR_PROS.resolve()}")

Pros:   0%|          | 0/49 [00:00<?, ?doc/s]

Generated 49 teacher schedule PDFs → /home/oscar/Projects/smartschool/fragenda/output/html/pros


## Locals (Rooms)

In [22]:
local_pdfs = generate_scope_pdfs(
    dataframe=df,
    scope="local",
    semver=SEMVER,
    template_dir=TEMPLATE_ROOT,
    template_name=TEMPLATE_NAME,
    output_dir=OUTPUT_DIR_LOCALS,
    renderer=RENDERER,
)
print(f"Generated {len(local_pdfs)} room schedule PDFs → {OUTPUT_DIR_LOCALS.resolve()}")

Locaux:   0%|          | 0/34 [00:00<?, ?doc/s]

Generated 34 room schedule PDFs → /home/oscar/Projects/smartschool/fragenda/output/html/locals


## Next steps
- Adjust the parameters cell to target a different export, version tag, or template variant.
- Review the PDFs inside the `output/html/` subdirectories and share them.
- Keep the original Excel-based flow for legacy consumers.

In [24]:
import shutil
from pathlib import Path
from pypdf import PdfWriter

release_tag = SEMVER.split('-')[-1]
release_root = (OUTPUT_DIR / f"horaires_{release_tag}").resolve()
if release_root.exists():
    shutil.rmtree(release_root)
release_root.mkdir(parents=True, exist_ok=True)

scopes = [
    (class_pdfs, 'classes'),
    (pro_pdfs, 'pros'),
    (local_pdfs, 'locaux'),
]

for pdf_list, scope in scopes:
    if not pdf_list:
        continue
    folder_name = f"horaire_{scope}-{SEMVER}"
    dest_dir = release_root / folder_name
    dest_dir.mkdir(parents=True, exist_ok=True)

    writer = PdfWriter()
    for pdf_path in sorted(pdf_list):
        pdf_path = Path(pdf_path)
        shutil.copy2(pdf_path, dest_dir / pdf_path.name)
        writer.append(str(pdf_path))
    combined_pdf = release_root / f"{folder_name}.pdf"
    with combined_pdf.open('wb') as fh:
        writer.write(fh)

zip_target = release_root.with_name(f"horaires_{release_tag}.zip")
if zip_target.exists():
    zip_target.unlink()
shutil.make_archive(str(zip_target.with_suffix("")), 'zip', release_root)

print(f"Release folder → {release_root}")
print(f"Archive created → {zip_target}")


Release folder → /home/oscar/Projects/smartschool/fragenda/output/html/horaires_v0.9
Archive created → /home/oscar/Projects/smartschool/fragenda/output/html/horaires_v0.9.zip


In [ ]:
import shutil
from pathlib import Path
from pypdf import PdfWriter
from tqdm.auto import tqdm

release_tag = SEMVER.split('-')[-1]
release_root = (OUTPUT_DIR / f"horaires_{release_tag}").resolve()
if release_root.exists():
    shutil.rmtree(release_root)
release_root.mkdir(parents=True, exist_ok=True)

scopes = [
    (class_pdfs, 'classes'),
    (pro_pdfs, 'pros'),
    (local_pdfs, 'locaux'),
]

for pdf_list, scope in scopes:
    if not pdf_list:
        print(f"Skipping {scope}, no PDFs")
        continue
    folder_name = f"horaire_{scope}-{SEMVER}"
    dest_dir = release_root / folder_name
    dest_dir.mkdir(parents=True, exist_ok=True)

    writer = PdfWriter()
    print(f"↳ Packaging {scope} ({len(pdf_list)} files)")
    for pdf_path in tqdm(sorted(pdf_list), unit="pdf"):
        pdf_path = Path(pdf_path)
        shutil.copy2(pdf_path, dest_dir / pdf_path.name)
        writer.append(str(pdf_path))
    combined_pdf = release_root / f"{folder_name}.pdf"
    with combined_pdf.open('wb') as fh:
        writer.write(fh)

zip_target = release_root.with_name(f"horaires_{release_tag}.zip")
if zip_target.exists():
    zip_target.unlink()
shutil.make_archive(str(zip_target.with_suffix("")), 'zip', release_root)

print(f"Release folder → {release_root}")
print(f"Archive created → {zip_target}")
